In [ ]:
################# - test_run_parameters - ##################
q_values = ["0.01", "0.25",  "0.5", "0.75", "0.90", "0.95", "0.99"]
q_default = "0.99"

ds_names = ["planetlaball", "seattleall"]
ds_default = "planetlaball"

e_values = ["0.5", "1", "1.5", "2", "2.5", "3", "3.5", "4", "4.5", "5"]
e_default = ["1.0", "2.0", "3.0", "4.0", "5.0"]

s_base = 16033099
s_step = 127
reps = 100
############################################################

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
import scipy.stats as stats
import math

In [ ]:
def read_realds_data(algo):
    distro_data = [pd.read_csv(f'Test_of_{algo}/test_real_ds_{ds_names[0]}/test_ds_{ds_names[0]}_e_0.5_1.csv')]
    distro_data[0]['std'] = 0.0
    distro_data[0]['cil'] = 0.0
    distro_data[0]['cir'] = 0.0
    distro_data[0]['d'] = 0
    for i in range(0,len(ds_names)):
        distro_data.append(pd.DataFrame(columns=distro_data[0].columns))
        for j in e_values:
            data_j = pd.read_csv(f'Test_of_{algo}/test_real_ds_{ds_names[i]}/test_ds_{ds_names[i]}_e_{j}_1.csv')
            for k in range(2,reps+1):
                data_j = pd.concat([pd.read_csv(f'Test_of_{algo}/test_real_ds_{ds_names[i]}/test_ds_{ds_names[i]}_e_{j}_{k}.csv'), data_j], ignore_index=True)
            data_j['d'] = i
            data_j = pd.concat([data_j, pd.DataFrame([data_j.mean()])], ignore_index=True)
            data_j['std'] = data_j['nae'].std()
            ci = stats.t.interval(0.95, df=reps-1, loc = data_j['nae'].mean(), scale = data_j['nae'].std()/math.sqrt(reps))
            data_j['cil'] = ci[0]
            data_j['cir'] = ci[1]
            data_j.drop(index=range(0,reps), inplace=True)
            distro_data[i+1] = pd.concat([distro_data[i+1], data_j], ignore_index=True)
            distro_data[i+1] = distro_data[i+1].astype({'d': 'int8'})
    distro_data.pop(0)
    distro_data = pd.concat(distro_data, ignore_index=False)
    distro_data.index = pd.Index(range(1,len(distro_data)+1))
    return distro_data
            

In [ ]:
def read_realq_data(algo):
    q_data = [pd.read_csv(f'Test_of_{algo}/test_real_q_0.01/test_q_0.01_e_1.0_ds_{ds_names[0]}_1.csv')]
    q_data[0]['std'] = 0.0
    q_data[0]['cil'] = 0.0
    q_data[0]['cir'] = 0.0
    q_data[0]['d'] = 0
    for i in range(0, len(q_values)):
        q_data.append(pd.DataFrame(columns=q_data[0].columns))
        for ds in range(0,len(ds_names)):
            for j in e_default:
                data_j = pd.read_csv(f'Test_of_{algo}/test_real_q_{q_values[i]}/test_q_{q_values[i]}_e_{j}_ds_{ds_names[ds]}_1.csv')
                for k in range(2,reps+1):
                    data_j = pd.concat(
                        [pd.read_csv(f'Test_of_{algo}/test_real_q_{q_values[i]}/test_q_{q_values[i]}_e_{j}_ds_{ds_names[ds]}_{k}.csv'), data_j], 
                        ignore_index=True
                    )
                data_j['d'] = ds
                data_j = data_j.astype({'d': 'int'})
                data_j = pd.concat([data_j, pd.DataFrame([data_j.mean()])], ignore_index=True)
                data_j['std'] = data_j['nae'].std()
                ci = stats.t.interval(0.95, df=reps-1, loc = data_j['nae'].mean(), scale = data_j['nae'].std()/math.sqrt(reps))
                data_j['cil'] = ci[0]
                data_j['cir'] = ci[1]            
                data_j.drop(index=range(0,reps), inplace=True)
                q_data[i+1] = pd.concat([q_data[i+1], data_j], ignore_index=True)
                q_data[i+1] = q_data[i+1].astype({'d': 'int8'})
    q_data.pop(0)
    q_data = pd.concat(q_data, ignore_index=False)
    q_data.index = pd.Index(range(1,len(q_data)+1))
    return q_data

In [5]:
for algo in ['ezq-sw', 'ldpq', 'frugal1u-rr', 'frugal2u-sw']:
    read_realds_data(algo).to_csv(f'test_real_on_ds_{algo}.csv')
    read_realq_data(algo).to_csv(f'test_real_on_q_{algo}.csv')